# Taller Práctico 01 — Del dato crudo a la decisión
### Dataset C — Movilidad urbana (sensores IoT de tráfico y clima, Medellín)

**Maestría en Ciencia de Datos y Analítica — Universidad EAFIT**
Fundamentos en Ciencia de Datos | Docente: Jorge Iván Padilla-Buriticá 

**Equipo:**

- Miguel Ángel Cano Salinas
- Daniel Correa Botero
- Daniel Felipe Arango Guarín

---

In [44]:
# Install dependencies
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [46]:
df_limpio = pd.read_csv('../data/raw/movilidad_sensores_LIMPIO.csv')
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1440 non-null   str    
 1   ubicacion         1440 non-null   str    
 2   tipo_via          1440 non-null   str    
 3   timestamp         1440 non-null   str    
 4   conteo_vehiculos  1440 non-null   int64  
 5   temperatura_c     1440 non-null   float64
 6   condicion_clima   1440 non-null   str    
 7   lat               1440 non-null   float64
 8   lon               1440 non-null   float64
dtypes: float64(3), int64(1), str(5)
memory usage: 168.9 KB


## Inventario de variables - Dataset limpio

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [47]:
df_contaminado = pd.read_csv('../data/raw/movilidad_sensores_CONTAMINADO.csv')
df_contaminado.info()

<class 'pandas.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1455 non-null   str    
 1   ubicacion         1455 non-null   str    
 2   tipo_via          1455 non-null   str    
 3   timestamp         1455 non-null   str    
 4   conteo_vehiculos  1310 non-null   float64
 5   temperatura_c     1383 non-null   float64
 6   condicion_clima   1368 non-null   str    
 7   lat               1455 non-null   float64
 8   lon               1455 non-null   float64
dtypes: float64(4), str(5)
memory usage: 170.2 KB


## Inventario de variables - Dataset contaminado

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [48]:
df_json = pd.read_json('../data/raw/clima_api_log.json')
df_json_flat = pd.concat(
    [
        df_json.drop(columns=['location', 'weather']).reset_index(drop=True),
        pd.json_normalize(df_json['location']),
        pd.json_normalize(df_json['weather'])
    ],
    axis=1
)
df_json_flat.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   request_id    120 non-null    str           
 1   timestamp     120 non-null    datetime64[us]
 2   sensor_id     120 non-null    str           
 3   lat           120 non-null    float64       
 4   lon           120 non-null    float64       
 5   temp_c        116 non-null    float64       
 6   condition     120 non-null    str           
 7   humidity_pct  120 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 9.6 KB


## Inventario de variables - Dataset JSON

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `request_id` | Nominal | JSON anidado | Semi-estructurada |
| `timestamp` | Fecha | JSON anidado | Semi-estructurada |
| `sensor_id` | Nominal | JSON anidado | Semi-estructurada |
| `lat` | Geoespacial | JSON anidado | Semi-estructurada |
| `lon` | Geoespacial | JSON anidado | Semi-estructurada |
| `temp_c` | Continuo | JSON anidado | Semi-estructurada |
| `condition` | Nominal | JSON anidado | Semi-estructurada |
| `humidity_pct` | Continuo | JSON anidado | Semi-estructurada |

## Reflexión

**¿Qué información se pierde o se distorsiona al forzar una fuente no estructurada/semi-estructurada dentro de una tabla rectangular?**

Se pueden perder la estandarización de los tipos de datos. La relación y jerarquía entre los campos, los cuales pueden brindar información adicional sobre la estructura de los datos. Además, el JSON puede agregar un campo nuevo mañana sin romperse a comparación del CSV. La tabla rectangular exige que todas las filas tengan las mismas columnas, así que cualquier campo y/o opcional nuevo obliga a rellenar con nulos todo lo anterior. 

## Tarea 2 — Diagnóstico GIGO

Functiones utilizadas en la etapa de pre-procesamiento para encontrar problemas en los datos

In [49]:
# Functions used to identify the data quality issues in the dataset

print("Null values per column:")
print(df_contaminado.isnull().sum())

print("\nUnique values in 'condicion_clima':")
print(df_contaminado["condicion_clima"].unique().value_counts())

print("\nNumber of duplicated rows based on 'sensor_id' and 'timestamp':")
print(df_contaminado.duplicated(subset=["sensor_id","timestamp"]).sum())

print("\nInvalid timestamps:")
print(pd.to_datetime(df_contaminado["timestamp"], errors="coerce").isna().sum())

print("\nNegative values in 'conteo_vehiculos':")
print((df_contaminado["conteo_vehiculos"] < 0).sum())

display(df_contaminado["lon"].describe())
print("\nLatitude and Longitude values outside Medellín's expected range ")
print((~df_contaminado["lat"].between(6.09, 6.39)).sum() & (~df_contaminado["lon"].between(-75.7, -75.47)).sum())

print("\nPlaceholder values (999) in 'conteo_vehiculos':")
print((df_contaminado["conteo_vehiculos"] >= 999).sum())


Null values per column:
sensor_id             0
ubicacion             0
tipo_via              0
timestamp             0
conteo_vehiculos    145
temperatura_c        72
condicion_clima      87
lat                   0
lon                   0
dtype: int64

Unique values in 'condicion_clima':
Soleado     1
Nublado     1
Sol         1
Lluvia      1
Nubes       1
soleado     1
LLUVIA      1
SOLEADO     1
lluvia      1
nublado     1
lluvioso    1
Name: count, dtype: int64

Number of duplicated rows based on 'sensor_id' and 'timestamp':
14

Invalid timestamps:
90

Negative values in 'conteo_vehiculos':
6


count    1455.000000
mean      -68.768157
std        22.612127
min       -75.591600
25%       -75.585105
50%       -75.572430
75%       -75.565360
max         6.287500
Name: lon, dtype: float64


Latitude and Longitude values outside Medellín's expected range 
121

Placeholder values (999) in 'conteo_vehiculos':
4


Hicimos un análisis exploratorio sobre el dataset que escogimos que es el de **Tráfico** y encontramos varios problemas que pueden afectar totalmente a la hora de analizar y mirar los resultados arrojados.

| Problema | Columna(s) | Cómo lo detectamos | Por qué nos importa |
|---|---|---|---|
| El clima está escrito de 11 formas para 3 valores reales (Sol, soleado, SOLEADO...) | `condicion_clima` | `df["condicion_clima"].value_counts(dropna=False)` | Si el clima queda dividido en etiquetas que son lo mismo pero escrito de forma diferente, tendremos la información segmentada, con valores asociados que no son precisos. |
| 14 entradas tienen el mismo sensor y timestamp | `sensor_id`, `timestamp` | `df.duplicated(subset=["sensor_id","timestamp"]).sum()` | Un mismo sensor en la misma unidad de tiempo tiene varias lecturas y esto no tiene sentido en el mundo real. Por ende, lo que nos proporciona es información inconsistente. No se hace un análisis completamente duplicadas, ya que son un subconjunto de los resultados de este análisis.
| Fechas en cinco formatos distintos y 90 de ellas no se pueden convertir | `timestamp` | `pd.to_datetime(df["timestamp"], errors="coerce").isna().sum()` | Si no tenemos el timestamp bien definido, no mediremos correctamente los datos en términos de horas pico. Por ende, se deberían convertir las que sean posibles, y el resto se eliminarían porque no aportan información relevante.  |
| 6 conteos vehiculares por debajo de 0 | `conteo_vehiculos` | `(df["conteo_vehiculos"] < 0).sum()` | Un conteo de vehículos no puede ser negativo. Esto es una falla del sensor que distorsiona cualquier medida. |
| 121 filas de lat y lon fuera del rango de Medellín | `lat`, `lon` | `(~df_contaminado["lat"].between(6.09, 6.39)).sum() & (~df_contaminado["lon"].between(-75.7, -75.47)).sum()` | Al comparar contra el rango de coordenadas de Medellín, esas entradas quedan por fuera y ubicarían el sensor en un lugar equivocado. |
| 4 lecturas valores extremadamente altos (`>= 999`) | `conteo_vehiculos` | `(df["conteo_vehiculos"] >= 999).sum()` | El 99999 dispara la media de 19 a 328, haciendo que los datos no reflejen la realidad del tráfico. | Datos nulos: son 145, 72 y 87 respectivamente | `conteo_vehiculos`, `temperatura_c`, `condicion_clima` | `df.isnull().sum()` | El conteo es algo fundamental en esta fuente de datos, si faltan lecturas, un pico real puede pasar desapercibido o podría alterar el resultado final.

# Tarea 3 - Transformación y limpieza con pandas

Corregimos los problemas utilizando la siguiente estructura: Primero quitamos los duplicados de filas que son duplicados exactos, luego nos encargamos de manejar el tema de que las fechas estaban duplicadas y que por consecuencia los eventos también. Despues nos encargamos de manejar el tema de valores imposibles y las coordenadas de Medellín que estaban desbordadas, para finalmente hacer la imputación de valores nulos cuando ya tenemos el resto del dataset estable y confiable.

In [50]:
df_contaminado_copy = df_contaminado.copy() #create a copy of the original to not lose the original data

df_contaminado_copy

,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700
1453,SEN06,Circular 4ta,Local,"18 de March de 2025, 20:00",12.0,23.2,Soleado,6.24469,-75.58500


# Etiquetas inconsistentes en condición climática

Encontramos etiquetas inconsistentes en la columna de condicion climática, la diferencia consiste en que su significado es equivalente pero están escritas de forma diferente.

**Decisión tomada:** Estandarización mediante mapeo (no eliminación).

Deecidimos estandarizar porque el problema real es que se tienen diferentes formatos para representar la misma categoría climática, por ejemplo "Sol", "Soleado" y "soleado" hacen referencía a la categoría "Soleado", decidimos forzarlo todo a esta categoría.

In [51]:
mapa_clima = {
    "sol": "Soleado", "soleado": "Soleado",
    "nubes": "Nublado", "nublado": "Nublado",
    "lluvia": "Lluvia", "lluvioso": "Lluvia",
}

df_contaminado_copy["condicion_clima"] = df_contaminado_copy["condicion_clima"].str.strip().str.lower().map(mapa_clima)

df_contaminado_copy["condicion_clima"].unique().value_counts()


Soleado    1
Nublado    1
Lluvia     1
Name: count, dtype: int64

# Sensor ID y Timestamp que se repiten

**Llave de negocio utilizada:** `sensor_id` + `timestamp`

**¿Por qué?**  Un sensor físico solo puede registrar una lectura en un momento específico. Si encontramos múltiples registros con el mismo `sensor_id` y `timestamp`, solo uno puede ser válido. Usar `.duplicated()` sin especificar subset detectaría únicamente filas 100% idénticas, pero no capturaría el problema de negocio real: un sensor no puede estar en dos lugares o medir dos conteos diferentes en el mismo instante.

**Decisión:** Se mantiene el último registro (`keep="last"`) para mantener un registro.

In [52]:
df_contaminado_copy = df_contaminado_copy.drop_duplicates(subset=["sensor_id","timestamp"], keep="last")
df_contaminado_copy

,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700
1453,SEN06,Circular 4ta,Local,"18 de March de 2025, 20:00",12.0,23.2,Soleado,6.24469,-75.58500


# Fechas en formatos diferentes

**Decisión tomada:** Eliminar filas con timestamps inválidos (no imputar).

**¿Por qué?**  El `timestamp` no es un valor que se pueda estimar de manera razonable. Dejandonos con un dato que no sirve para analítica horaria (horas picos, fines de semana, día/noche, etc...)


La función `pd.to_datetime()` con `errors="coerce"` intenta parsear múltiples formatos automáticamente y convierte a `NaT` (Not a Time) los que no puede interpretar. Estas 90 filas se eliminan porque sin tiempo válido, los datos pierden su contexto temporal fundamental.

In [53]:
invalid_entries = pd.to_datetime(df_contaminado_copy["timestamp"], errors="coerce").isna()
print("Fechas que no se pueden convertir:", invalid_entries.sum(),
      f"({invalid_entries.mean():.1%} del total)")

df_contaminado_copy["timestamp"] = pd.to_datetime(df_contaminado_copy["timestamp"], errors="coerce")
df_contaminado_copy = df_contaminado_copy.dropna(subset=["timestamp"])

df_contaminado_copy


Fechas que no se pueden convertir: 90 (6.2% del total)


,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1449,SEN02,Av. 33,Arteria,2025-03-08 18:00:00,34.0,22.7,Soleado,6.21439,-75.57296
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700


# Conteo Vehiculares Negativos

**Decisión tomada:** Convertir a `NaN`, no eliminar la fila completa.

**¿Por qué?** Un conteo negativo es claramente un error, pero el resto de los datos de esa fila (ubicación, timestamp, temperatura, clima) siguen siendo válidos. Eliminar toda la fila perdería información geoespacial y temporal útil. Al convertir solo el conteo a `NaN`, permitimos que sea imputado posteriormente con la mediana.

In [54]:
negative_counts = (df_contaminado_copy["conteo_vehiculos"] < 0)
df_contaminado_copy.loc[negative_counts, "conteo_vehiculos"] = np.nan

# Coordenadas fuera de la ciudad

**Decisión tomada:** Descartar filas con coordenadas fuera de rango (no corregir automáticamente).

**¿Cómo definimos el rango geográfico válido?** Se consultaron fuentes web oficiales que establecen que Medellín se encuentra aproximadamente entre:
- Latitud: 6.09° N a 6.39° N
- Longitud: -75.7° W a -75.47° W

**¿Por qué?** No tenemos forma confiable de saber dónde debería estar realmente el sensor.

Intentar "corregir" automáticamente (por ejemplo, intercambiando lat y lon) introduciría suposiciones no verificables. Es más seguro descartar estas 121 filas y trabajar solo con datos geográficamente verificables.

In [ ]:
out_of_range = (~df_contaminado_copy["lat"].between(6.09, 6.39)) | (~df_contaminado_copy["lon"].between(-75.7, -75.47))
df_contaminado_copy = df_contaminado_copy[~out_of_range]

# Valores extremadamente altos

**Decisión tomada:** Convertir valores ≥ 999 a `NaN` (tratarlos como nulos).

**¿Por qué 999 es sospechoso?** En el contexto de los conteos observados son valores que no tienen sentido, ya que en el análisis exploratorio detectamos que estos valores extremos disparan la media de 19 a 328 vehículos, distorsionando completamente las métricas.

**¿Por qué NaN y no eliminar?** Similar al caso de conteos negativos, el resto de la información de la fila (ubicación, timestamp, clima) sigue siendo válida. Convertir a `NaN` permite que estos valores sean imputados posteriormente.

In [ ]:
df_contaminado_copy.loc[df_contaminado["conteo_vehiculos"] >= 999, "conteo_vehiculos"] = np.nan

# Datos nulos

**Estrategia de imputación aplicada:**

**1. `conteo_vehiculos` → Mediana con contexto (no media ni eliminación)**

- **¿Por qué mediana y no media?** La media es muy sensible a outliers. Si un sensor tuvo lecturas atípicas (accidente, evento especial), la media se distorsionaría. La mediana es más robusta y representa mejor el "valor típico".
- **¿Por qué contextual (sensor + hora)?** El tráfico varía dramáticamente por ubicación y hora. Primero intentamos imputar con la mediana del mismo sensor a la misma hora del día (ej: sensor S1 a las 8am), porque refleja el patrón real de ese punto. Si no hay datos suficientes, usamos la mediana general del sensor como respaldo.
- **¿Por qué no eliminar?** Perderíamos 145+ filas valiosas. La imputación con contexto temporal/espacial es metodológicamente válida en datos de sensores con patrones predecibles.

**2. `temperatura_c` → Mediana por sensor (no media ni moda)**

- **¿Por qué mediana y no media?** Protege contra valores extremos anómalos que podrían haber quedado en el dataset.
- **¿Por qué por sensor y no global?** Cada sensor está en una ubicación geográfica distinta (valle, ladera, urbano denso) con microclimas propios. La temperatura promedio de un sensor en El Poblado puede ser 3-5°C diferente a uno en Robledo. La mediana por sensor es más representativa que la temperatura global del dataset.
- **¿Por qué no eliminar?** La temperatura es secundaria para el análisis de tráfico. Mantener las filas permite estudiar patrones de movilidad incluso sin dato climático.

**3. `condicion_clima` → Valor centinela "Desconocido" (no moda ni eliminación)**

- **¿Por qué "Desconocido" y no la moda?** Imputar con la moda (probablemente "Soleado" en Medellín) introduciría información falsa. Es mejor ser explícitos: **no sabemos** el clima en ese momento. Esto es más honesto científicamente.
- **¿Por qué no eliminar?** El resto de la información de tráfico sigue siendo útil. Muchos análisis (conteos por hora, patrones por ubicación) no requieren la condición climática.
- **Ventaja del centinela:** Permite **filtrar** estas filas en análisis que sí requieren clima válido, sin perder los datos para otros propósitos. Es trazable y reversible.

In [ ]:
# 1) conteo_vehiculos → mediana del mismo sensor a esa misma hora, con respaldo por sensor
df_contaminado_copy["conteo_vehiculos"] = df_contaminado_copy["conteo_vehiculos"].fillna(
    df_contaminado_copy.groupby([df_contaminado_copy["sensor_id"], df_contaminado_copy["timestamp"].dt.hour])["conteo_vehiculos"].transform("median")
).fillna(df_contaminado_copy.groupby("sensor_id")["conteo_vehiculos"].transform("median"))

# 2) temperatura_c → mediana por sensor
df_contaminado_copy["temperatura_c"] = df_contaminado_copy["temperatura_c"].fillna(
    df_contaminado_copy.groupby("sensor_id")["temperatura_c"].transform("median")
)

# 3) condicion_clima → categoría explícita "Desconocido"
df_contaminado_copy["condicion_clima"] = df_contaminado_copy["condicion_clima"].fillna("Desconocido")

print("Nulos por columna:\n", df_contaminado_copy[["conteo_vehiculos", "temperatura_c", "condicion_clima"]].isnull().sum())

Nulos por columna:
 conteo_vehiculos    0
temperatura_c       0
condicion_clima     0
dtype: int64


In [ ]:
# Functions used to identify the data quality issues in the dataset

print("Null values per column:")
print(df_contaminado_copy.isnull().sum())

print("\nUnique values in 'condicion_clima':")
print(df_contaminado_copy["condicion_clima"].unique().value_counts())

print("\nNumber of duplicated rows based on 'sensor_id' and 'timestamp':")
print(df_contaminado_copy.duplicated(subset=["sensor_id","timestamp"]).sum())

print("\nInvalid timestamps:")
print(pd.to_datetime(df_contaminado_copy["timestamp"], errors="coerce").isna().sum())

print("\nNegative values in 'conteo_vehiculos':")
print((df_contaminado_copy["conteo_vehiculos"] < 0).sum())

print("\nLatitude and Longitude values outside Medellín's expected range ")
print((~df_contaminado_copy["lat"].between(6.09, 6.39)).sum() & (~df_contaminado_copy["lon"].between(-75.7, -75.47)).sum())

print("\nPlaceholder values (999) in 'conteo_vehiculos':")
print((df_contaminado_copy["conteo_vehiculos"] >= 999).sum())


Null values per column:
sensor_id           0
ubicacion           0
tipo_via            0
timestamp           0
conteo_vehiculos    0
temperatura_c       0
condicion_clima     0
lat                 0
lon                 0
dtype: int64

Unique values in 'condicion_clima':
Soleado        1
Nublado        1
Lluvia         1
Desconocido    1
Name: count, dtype: int64

Number of duplicated rows based on 'sensor_id' and 'timestamp':
0

Invalid timestamps:
0

Negative values in 'conteo_vehiculos':
0

Latitude and Longitude values outside Medellín's expected range 
0

Placeholder values (999) in 'conteo_vehiculos':
0
